# Bicycle JPEG timing prototype
This is an engineering benchmark on 100 COCO development images. It measures extraction, disk usage, and a scoring-only update on cached features. The formal A/B study, test set and decision rule are outside this run.

In [ ]:
from time import perf_counter
start = perf_counter()
import json
import io
import os
import random
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_XET"] = "1"
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "transformers==4.57.1"], check=True)
import numpy as np
import torch
import transformers
from torch import nn
from torchvision.ops import box_convert, box_iou
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from huggingface_hub import model_info

assert torch.cuda.is_available(), "This benchmark requires a Kaggle GPU."
torch.manual_seed(2026)
device = "cuda:0"
output_dir = Path("/kaggle/working")
cache = output_dir / "features"
cache.mkdir(exist_ok=True)
report = {"gpu": torch.cuda.get_device_name(0), "torch": str(torch.__version__),
          "transformers": transformers.__version__, "prompt": "bicycle.", "qualities": [95, 90]}
print(report, flush=True)

In [ ]:
# Stay on the mounted Kaggle dataset. Reserve these source IDs for development.
mount = Path("/kaggle/input")
annotation_file = next(mount.rglob("instances_val2017.json"))
coco_root = annotation_file.parent.parent
data = json.loads(annotation_file.read_text())
bicycle_id = next(cat["id"] for cat in data["categories"] if cat["name"] == "bicycle")
from collections import defaultdict
boxes_by_image = defaultdict(list)
crowd = set()
for ann in data["annotations"]:
    if ann["category_id"] == bicycle_id:
        if ann["iscrowd"]:
            crowd.add(ann["image_id"])
        else:
            boxes_by_image[ann["image_id"]].append(ann["bbox"])

eligible = [im for im in data["images"] if im["id"] not in crowd]
rng = random.Random(2026)
selected = []
for positive in (True, False):
    pool = [im for im in eligible if bool(boxes_by_image[im["id"]]) == positive]
    selected.extend(rng.sample(sorted(pool, key=lambda im: im["id"]), 50))
rng.shuffle(selected)
manifest = [{"image_id": im["id"], "filename": im["file_name"],
             "boxes_xywh": boxes_by_image[im["id"]]} for im in selected]
(output_dir / "development_manifest.json").write_text(json.dumps(manifest, indent=2))
report["sources"] = len(selected)
report["positive_sources"] = sum(bool(row["boxes_xywh"]) for row in manifest)
report["annotation_split"] = "COCO val2017 development subset; exclude from future confirmatory testing"
print("Selected:", report["sources"], flush=True)

In [ ]:
model_id = "IDEA-Research/grounding-dino-tiny"
revision = model_info(model_id).sha
report["checkpoint"] = f"{model_id}@{revision}"
processor = AutoProcessor.from_pretrained(model_id, revision=revision, use_fast=False)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id, revision=revision).to(device).eval()
model.requires_grad_(False)
report["setup_seconds"] = perf_counter() - start

def extract_one(rgb, quality):
    encoded = io.BytesIO()
    rgb.save(encoded, format="JPEG", quality=quality, subsampling=2, optimize=False)
    encoded.seek(0)
    with Image.open(encoded) as view:
        inputs = processor(images=view.convert("RGB"), text="bicycle.", return_tensors="pt").to(device)
    with torch.inference_mode():
        pred = model(**inputs)
        queries = pred.intermediate_hidden_states[:, -1]
        text = pred.encoder_last_hidden_state_text
        # Restrict the engineering readout to the prompt word, excluding special tokens and punctuation.
        tokens = processor.tokenizer.convert_ids_to_tokens(inputs.input_ids[0].tolist())
        word = tokens.index("bicycle")
        scores = pred.logits[0, :, word]
        reconstructed = (queries @ text.transpose(-1, -2))[0, :, word]
        torch.testing.assert_close(reconstructed, scores, atol=0.001, rtol=0.0001)
        return {"queries": queries[0].half().cpu(), "word_embedding": text[0, word].float().cpu(),
                "boxes_cxcywh": pred.pred_boxes[0].float().cpu(), "logits": scores.float().cpu()}

def load_rgb(im):
    with Image.open(coco_root / "val2017" / im["file_name"]) as image:
        return image.convert("RGB")

warm_start = perf_counter()
extract_one(load_rgb(selected[0]), 95)
torch.cuda.synchronize()
report["warmup_seconds"] = perf_counter() - warm_start
torch.cuda.reset_peak_memory_stats()
extract_start = perf_counter()
for n, im in enumerate(selected, 1):
    rgb = load_rgb(im)
    for quality in report["qualities"]:
        record = extract_one(rgb, quality)
        torch.save(record, cache / f"{im['id']}_{quality}.pt")
    if n % 20 == 0:
        print(f"{n}/100 sources cached", flush=True)
torch.cuda.synchronize()
report["extraction_seconds"] = perf_counter() - extract_start
files = list(cache.glob("*.pt"))
assert len(files) == 200
report["views"] = len(files)
report["cache_bytes"] = sum(file.stat().st_size for file in files)
report["peak_allocated_gpu_bytes"] = torch.cuda.max_memory_allocated()
report["projected_extraction_hours_14000_views"] = report["extraction_seconds"] * 70 / 3600
print("Extraction seconds:", report["extraction_seconds"], flush=True)

In [ ]:
# One engineering gradient update. This is deliberately separate from the planned A/B experiment.
# The target here is whether a frozen candidate covers any labelled bicycle at IoU >= 0.5.
examples = []
for im in selected:
    if not boxes_by_image[im["id"]]:
        continue
    record = torch.load(cache / f"{im['id']}_95.pt", weights_only=True)
    candidates = box_convert(record["boxes_cxcywh"], "cxcywh", "xyxy")
    gt = torch.tensor(boxes_by_image[im["id"]], dtype=torch.float32)
    gt /= torch.tensor([im["width"], im["height"], im["width"], im["height"]])
    labels = box_iou(candidates, box_convert(gt, "xywh", "xyxy")).amax(dim=1) >= 0.5
    if labels.any() and (~labels).any():
        examples.append((record, labels))
    if len(examples) == 4:
        break
assert examples, "No covered bicycles available for head update."
q = torch.stack([r["queries"] for r, _ in examples]).float().to(device)
word = torch.stack([r["word_embedding"] for r, _ in examples]).to(device)
targets = torch.stack([labels for _, labels in examples]).float().to(device)
frozen_logits = torch.stack([r["logits"] for r, _ in examples]).to(device)
head = nn.Sequential(nn.Linear(q.shape[-1], 32), nn.ReLU(), nn.Linear(32, q.shape[-1])).to(device)
nn.init.zeros_(head[-1].weight)
nn.init.zeros_(head[-1].bias)
def score():
    return torch.einsum("bqd,bd->bq", q + head(q), word)
before = score()
base = torch.einsum("bqd,bd->bq", q, word)
torch.testing.assert_close(before, base)
report["max_probability_cache_roundtrip_error"] = (base.sigmoid() - frozen_logits.sigmoid()).abs().max().item()
assert report["max_probability_cache_roundtrip_error"] < 0.005
loss_fn = nn.BCEWithLogitsLoss(pos_weight=(1-targets).sum()/targets.sum())
loss = loss_fn(before, targets)
optimizer = torch.optim.AdamW(head.parameters(), lr=0.0001)
loss.backward()
optimizer.step()
after = score().detach()
assert torch.isfinite(after).all() and (after - before.detach()).abs().max() > 0
report["head_smoke"] = {"sources": len(examples), "covered_query_targets": int(targets.sum().item()),
                        "loss_before": loss.item(), "loss_after": loss_fn(after, targets).item(),
                        "max_logit_change": (after - before.detach()).abs().max().item()}
report["total_seconds_from_first_cell"] = perf_counter() - start
(output_dir / "prototype_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))